# Perioperative Digital Twin — GPU training (Colab)

Trains the PyTorch tabular deep net (and the sklearn baselines) on a **CUDA GPU**, then downloads the weights + metrics to load back into the app.

**Runtime → Change runtime type → GPU** before running.


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')


## 1. Dependencies

In [ ]:
!pip -q install vitaldb pandas pyarrow scikit-learn certifi


## 2. Get the code
Set `REPO` to your GitHub URL. If the repo is private, use a token URL:
`https://<TOKEN>@github.com/<user>/<repo>.git`


In [ ]:
REPO = 'https://github.com/junaidraza343/AI-Digital-Twin-for-Perioperative-Risk-Prediction-Drug-Response-Simulation.git'
!git clone $REPO twin_repo || (cd twin_repo && git pull)
%cd twin_repo


## 3. Build a dataset on Colab
VitalDB is open — this downloads the 1 Hz numeric tracks and builds the labeled windows. Pick `N_CASES` (bigger = better, slower).


In [ ]:
import os, certifi
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()
os.environ['TWIN_PROGRESS'] = '1'
N_CASES = 2000  # scale up on Colab
!python -m scripts.run_all_subset --n-cases $N_CASES


## 4. Train the deep net (GPU) + the sklearn baselines

In [ ]:
!python -m scripts.train_deepnet --epochs 60
import pandas as pd
from scripts.run_baseline import run
w = pd.read_parquet('results/windows.parquet'); m = pd.read_csv('results/eligible_cases.csv')
tbl = run(w, m)
dn = pd.read_csv('results/deepnet_results.csv')
full = pd.concat([tbl, dn], ignore_index=True)
print(full.to_string(index=False))


## 5. Download the trained artifacts
Load `deepnet.pt` + the CSVs back into your local repo's `results/` folder.


In [ ]:
from google.colab import files
for f in ['results/deepnet.pt','results/deepnet_results.csv',
          'results/selection_bias_results.csv','results/windows.parquet']:
    try: files.download(f)
    except Exception as e: print('skip', f, e)
